# Attention
At a high level, attention allows a model to incorporate context of a given word into its processing abilities; prior to attention, models processed a single token at a time. One quick example of why this is important is when we consider homonyms: if a model sees the word "rock", how does it know if we are talking about a stone? Or an adjective that means great? Or a genre of music?

## How it works
Let's walk through an example of a simple model with embeddings_dim = 4:
"The dog chased the squirrel".

```
The:      [.4, .3, -.7, .1]
Dog:      [.1, .6, ,2, -4]
Chased:   [-.7, .4, .2, -.1]
The:      [.4, .3, -.7, .1]
Squirrel: [.2, -.8, -.3, .4]
```
From our input, we want to create three vectors: a query vector, a key vector, and a value vector:

```
Q = x @ W_Q
K = x @ W_K
V = x @ W_V
```
where the weight matrices are initialized randomly and then learned through training.

We can think of the Query vector as asking a question: for a given token, which tokens are important? And the Key vector representing what each token has to offer.

### Attention Scores
The formula for attention scores is as follows:

$Score_{i,j} = Q_i ⋅ K_j$

which simply measures how well token j's key answers token i's query.

### Transformation of Attention Scores
After computing scores, there are some 'transformations' we will want to undergo.


**Masking**

Again consider our sentence: "the dog chased the squirrel". At the time that the model is processing the word "chased", it would be cheating if it knew what the dog was chasing (the squirrel). So we apply a triangular mask so the model is only allowed to use seen tokens as context:

```
Token | Context
  0   | 0
  1   | 0, 1
  2   | 0, 1, 2
  3   | 0, 1, 2, 3
  and so on
```

**Softmax**


We want to use these dot products to ultimately produce a weighted mixture of the words in the sequence as context for a given token. It might immediately seem correct to just divide by the sum of each row. But notice that we have applied 2 linear transformations to our input vector x. Since linear tranformations of linear transformation are just more linear transformations,  we need to apply a non-linear transformation to extract non-linear patterns in the data. A common method is to use the softmax function:

$\sigma(\mathbf{z})_i = \frac{e^{z_i}}{\sum_{j=1}^K e^{z_j}} \quad$ which exponentiates the values first before dividing by the sum. Note that we ultimately have weights that sum to 1, giving up acceptable weights.

**Scaling**

With high and low values, the softmax function approaches asymptotes. Our gradients essentially vanish and weights are thus not adjusted. As a solution to this problem, we scale the scores before applying softmax:

$Var(Q \cdot K) =$ embeddings_dim, so we can divide the scores by embeddings_dim:

$Var(\frac{Q \cdot K}{\sqrt{d}}) = \frac{Var(Q \cdot K)}{\sqrt{d}^2} = \frac{d}{d} = 1$ which keeps our variance stable.

### Output
After we apply softmax to get these weights, we need to multiply these weights by the value to get our mixed context final product:

```
output = softmax_weights @ V
```





# Multi-Headed Attention
The above information was not entirely accurate. In the method I described above, we create a representation of a given token as a mixture of its context. However, there are a lot of nuances and relationships that we need to account for. Consider a different sentence: "She washed her dirty hands". As humans, a few things we immediately internalize include:


* subject: "she"
* object: "hands"
* verb: "wash"
* tense: past
* possessive: "her"
* gender of possessive (as implied by subject): feminine
* adjective: "dirty"
* noun modified by adjective" "hands"

Even in this simple four word sentence, there is a lot going on. It is really hard to capture all of these syntactic relationships that we can immediately discern in one attention block.

## Solution: Multi-Headed Attention
Consider again the attention mechanism as described above:

```
                                      @ W_Q = Q (seq_len * 1)
x (seq_len * embeddings_dim) ->       @ W_K = K (seq_len * 1)   
                                      @ W_V = V (seq_len * 1)
                                      where weights are (embeddings_dim * 1)

Q * K -> scores (seq_len * seq_len)
scores / sqrt(embeddings_dim) -> scaled_scores
mask(scaled_scores) -> masked_scores
softmax(masked_scores) -> softmax_weights (seq_len * seq_len)
softmax_weights * V -> output (seq_len * 1)
```
In particular, note that our input is our embedding sequence length matrix of shape seq_len * embeddings_dim. With multi-headed attention, we continue to expose each head to the full set of embeddings but project them onto a space of shape seq_len * (embeddings_dim // head_count) as we now configure our weight matrices to be of shape (embeddings_dim * dim_per_head) where dim_per_head = embeddings_dim // head_count. In our single-head attention example, our weight matrices were of shape (embeddings_dim * 1). Consider an example with the query weight matrix using our original sentence and two heads:

```
                                           Head 1            Head 2
The:      [.4, .3, -.7, .1]   ->       [q000, q001]        [q010, q011]
Dog:      [.1, .6, .2, -4]    ->       [q100, q101]        [q110, q111]
Chased:   [-.7, .4, .2, -.1]  ->       [q200, q201]        [q210, q211]
The:      [.4, .3, -.7, .1]   ->       [q300, q301]        [q310, q311]
Squirrel: [.2, -.8, -.3, .4]  ->       [q400, q401]        [q410, q411]
```
and we repeat the attention mechanism FOR each head.
Revisiting our attention mechanism but now with head_count > 1, we see:

```
dim_per_head = embeddings_dim // head_count
                                    @ W_Q = Q (seq_len * dim_per_head)
x (seq_len * embeddings_dim) ->     @ W_K = K (seq_len * dim_per_head)   
                                    @ W_V = V (seq_len * dim_per_head)
                                    where weights are (embeddings_dim * dim_per_head)

Q * K -> scores (seq_len * seq_len)
scores / sqrt(dim_per_head) -> scaled_scores
mask(scaled_scores) -> masked_scores
softmax(masked_scores) -> softmax_weights (seq_len * seq_len)
softmax_weights * V -> output (seq_len * dim_per_head)
```
For each head, we now have output of shape (seq_len * dim_per_head). And since we have head_count heads (recall head_count * dim_per_had = embeddings_dim), we can concatenate each of the outputs to achieve one output matrix (of correct shape (seq_len * embeddings_dim) that aggregates all of the information from each of the separate heads!

As a last step, we multiply the output matrix by an output weight matrix of shape (embeddings_dim * embeddings_dim) to ensure that we mix the information across all of the heads-- they can communicate with each other. Otherwise, the outputs of head 1 would never communicate with the outputs of head 2, or any other head for that matter.

In comparing the above psuedocode to the actual code, notice that we actually left out two important pieces. The first is the rotation of the Q and K vectors, which we went over with the creation of RoPE class. The second is dropouts.

### Dropouts
To prevent overfitting (by means of over-relying on particular tokens), we exclude some percentage of the tokens in each pass (our dropout rate is 10%). We do so in two spots:
1. After softmax: this forces the model to not rely too heavily on any single token
2. Applied to the final output: this is general regularization on the output. It is called residual dropout because in the full transformer block, this output gets added to the input via a residual connection.

## Types of Attention
In "Attention is All You Need", we learn about three different implementations of attention:
1. Encoder self-attention: each token attends to all tokens in the input sequence, no mask (sentiment analysis)
2. Decoder self-attention: each token attends only to previous tokens (what we built)
3. Cross-attention: decoder queries attend to encoder keys/values (think about language translation models: this might look like the Query being in French, and Keys/Values in English; "given what I have generated, which parts of the English input should I focus on next?").

Since then, there have been many variants of attention developed. Right now, Group Query Attention (GQA) is the default for most open-weight models, and works by sharing keys and values across head groups to reduce memory without much accuracy loss.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# copying over RoPE code
class RoPE(nn.Module):
  def __init__(self, embeddings_dim, max_seq_len: int = 2048, theta: float = 10000.):
    super().__init__()

    # asserting model is even (0,1), (2,) -> this would be a problem
    assert(embeddings_dim % 2 == 0)

    # indices of the first dimension of each pair: 0, 2, 4, 6,...
    pair_number = torch.arange(0, (embeddings_dim // 2), 1).float()

    # compute frequencies, one for each pair
    denom = theta ** (2*pair_number / embeddings_dim)
    inv_freq = 1. / denom

    # compute position indices
    positions = torch.arange(0, max_seq_len, 1)

    # compute rotation angles for each position and each pair
    pair_angles = torch.outer(positions, inv_freq)

    # compute rotation angles for each position and each dimension
    # note the ordering: [0,0,1,1,2,2...]
    angles = torch.repeat_interleave(pair_angles, repeats=2, dim=-1)

    # compute sine and cosine
    sine = torch.sin(angles)
    cosine = torch.cos(angles)
    self.register_buffer("sine_cached", sine)
    self.register_buffer("cos_cached", cosine)

  @staticmethod
  def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """
    For a given pair (x0, x1), the rotation formula is as follows:
    x0' = x0*cos(theta) - x1*sin(theta)
    x1' = x0*sin(theta) + x1*cos(theta)
    We could loop through all pairs, but that would be inefficient.
    rotate_half = [-x1, x0, -x3, x2, -x5, x4,...]

    x' = x*cos(theta) + rotate_half(x)*sin(theta)

    x0' = x0*cos(theta) - x1*sin(theta)
    x1' = x0*sin(theta) + x1*cos(theta)
    x2' = x2*cos(theta) - x3*sin(theta)
    x3' = x2*sin(theta) + x3*cos(theta)
    """
    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]
    rotated = torch.stack((-x_odd, x_even), dim=-1).flatten(start_dim=-2)
    return rotated

  # now appling ROPE to a given Q or K vector
  def forward(self, x: torch.Tensor) -> torch.Tensor:

    # [batch, heads, seq, embeddings_dim]
    seq_len = x.shape[-2]

    # get cos and sine
    cos = self.cos_cached[:seq_len]
    sin = self.sine_cached[:seq_len]

    # broadcast
    # each position and each dimension gets its own RoPE angle,
    # but the same angle table is shared across batches and heads.
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)

    x_prime = x * cos + self.rotate_half(x) * sin
    return x_prime

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, embeddings_dim: int, head_count: int, dropout: float = .1):
    super().__init__()
    assert embeddings_dim % head_count == 0, "embeddings_dim must be divisible by head_count"
    dim_per_head = embeddings_dim // head_count

    self.head_count = head_count
    self.dim_per_head = dim_per_head
    self.dropout = dropout

    """ weight matrices for ONE head would be defined as:

    self.W_Q = nn.Linear(embeddings_dim, dim_per_head)
    self.W_K = nn.Linear(embeddings_dim, dim_per_head)
    self.W_V = nn.Linear(embeddings_dim, dim_per_head)

    but this is not efficient. Instead, we can project each matrix simultaneously, for all heads, to optimize for GPU

    """
    self.w_qkv = nn.Linear(embeddings_dim, 3*embeddings_dim, bias = False)
    self.w_o = nn.Linear(embeddings_dim, embeddings_dim, bias = False)
    self.rope = RoPE(dim_per_head)
    self.attn_dropout = nn.Dropout(dropout)
    self.resid_dropout = nn.Dropout(dropout)

  def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
    """
    recall x.shape: [batch, seq_len, embeddings_dim]
    """
    batch_size, seq_len, embeddings_dim = x.shape
    # project x into QKV space
    qkv = self.w_qkv(x)

    """
    Recall: QKV_W.shape = [batch, seq_len, 3*embeddings_dim]
    We want to split into Q, K, V (where shape of each [batch, seq_len, embeddings_dim])and further into each head.
    Ignoring batches when redistributing (they just stack sequences for parallel processing), we want
    our data to ultimately be shaped: [batch, head_count, seq_len, dim_per_head]
    """
    qkv_reshape = qkv.reshape(batch_size, seq_len, 3, self.head_count, self.dim_per_head)
    qkv = qkv_reshape.permute(2, 0, 3, 1, 4) # [3, batch_size, head_count, seq_len, dim_per_head]
    q, k, v = qkv[0], qkv[1], qkv[2]

    # apply rotation to q and k vectors
    q = self.rope(q)
    k = self.rope(k)

    # transformations
    scores = (q @k.transpose(-1, -2))
    scores_scaled = scores / math.sqrt(self.dim_per_head)
    if mask is not None:
      scores_scaled = scores_scaled.masked_fill(mask == 0, float('-inf'))
    softmax_weights = F.softmax(scores_scaled, dim=-1)
    softmax_weights_dropout = self.attn_dropout(softmax_weights)

    # output: shape is [batch, head_count, seq_len, dim_per_head]
    output = softmax_weights_dropout @ v

    # we want to concatenate the various heads s.t. output.shape = [batch, seq_len, embeddings_dim]
    output = output.transpose(1, 2).reshape(batch_size, seq_len, embeddings_dim)

    # output projection
    output = self.w_o(output)

    # dropout again
    output = self.resid_dropout(output)

    return output


In [ ]:
attn = MultiHeadAttention(head_count=2, embeddings_dim=4)
x = torch.randn(1, 10, 4)
output = attn(x)
output.shape

torch.Size([1, 10, 4])